In [ ]:
#| default_exp menus

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *

Define menu rows independently of the host-owned labels and key bindings.

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass

### The rows

`MenuItem` is a row naming an action. A blank action is a separator, which is why `action` has a
default. `kind` is advisory and reaches the host untouched; `warn` marks a row a caller may want to
draw as destructive.

In [ ]:
#| export
@dataclass(frozen=True)
class MenuItem:
    "One action in a menu. A blank action is a separator."
    action: str = ''
    kind: str = ''
    def __post_init__(self):
        if self.kind not in ('', 'warn'):
            raise ValueError(f'{self.action or "separator"}: bad menu kind {self.kind!r}')

In [ ]:
MenuItem('open_folder'), MenuItem()

In [ ]:
#| hide
test_eq(MenuItem().action, '')                     
test_eq(MenuItem('git_discard', 'warn').kind, 'warn')
test_fail(lambda: MenuItem('x', 'loud'), contains='bad menu kind')

`Std` is a row AppKit already implements. Cut, Copy, Paste and Select All are not the host's to
handle: the selector goes to the responder chain with a nil target, and whatever has focus answers.
`key` and `mods` spell the chord, because no keymap entry backs a row like this.

In [ ]:
#| export
@dataclass(frozen=True)
class Std:
    "A row AppKit implements itself. The selector goes to the responder chain, target nil."
    title: str
    selector: str
    key: str = ''
    mods: str = 'mod'

In [ ]:
#| export
@dataclass(frozen=True)
class Js:
    "A row no action backs, named by the expression it runs."
    title: str
    expr: str

In [ ]:
Std('Cut', 'cut:', 'x'), Js('Reload', 'location.reload()')

### Key bindings

`Binding` supplies the label, chord, scope, and bound state used by the native menu.

In [ ]:
#| export
@dataclass(frozen=True)
class Binding:
    "What a menu row reads from the host's keymap: the chords, the label, and where it applies."
    keys: tuple = ()
    label: str = ''
    scope: str = 'global'
    @property
    def bound(self): return bool(self.keys)

In [ ]:
keys = {'save': Binding(('mod+s',), 'Save'), 'find': Binding(('mod+f',), scope='editor')}
keys.get('save'), keys.get('nothing')

`bound` is a property rather than a field so that an unbound action is spelled `Binding()` and
cannot disagree with itself.

In [ ]:
#| hide
test_eq(Binding().bound, False)
test_eq(Binding(('mod+s',)).bound, True)
test_eq(Binding(('mod+s',)).scope, 'global')       
test_eq(keys.get('save').keys[0], 'mod+s')
test_is(keys.get('nothing'), None)                 

`window.menu_bar` takes the table and the lookup together. Nothing in kavacha ships a default for
either: a library that packages somebody else's app has no opinion about what belongs on their File
menu.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()